In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transp

# Longer dataframe

In [2]:
df_deg_bins <- fread("./interm/df_WB_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "30298932 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.75,27.50,T0DayMin1,1,0,1
42660,96.75,28.25,T0DayMin1,4,0,1


# Compute mean/max T per day preceding measurement

In [3]:
df_deg_bins_day <- df_deg_bins %>%
                        group_by(Measured_date_CDC,cell_x,cell_y, Day_Min)%>%
                        summarize(WB_min = min(Temp),
                                  WB_mean = sum(N_hours*Temp)/24,
                                  WB_max = max(Temp))
sprintf("%i x %i dataframe", nrow(df_deg_bins_day), ncol(df_deg_bins_day))
head(df_deg_bins_day,2)

`summarise()` has grouped output by 'Measured_date_CDC', 'cell_x', 'cell_y'. You can override using the `.groups` argument.


[1] "3801750 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,WB_min,WB_mean,WB_max
<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>
42022,74,15.25,1,19,22.00000,26
42022,74,15.25,2,18,21.45833,25


In [4]:
data.frame(df_deg_bins_day) %>% 
        select(WB_min,WB_mean,WB_max)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic     N     Mean  St. Dev.  Min   Pctl(25) Median Pctl(75)  Max 
------------------------------------------------------------------------
WB_min    3,801,750 17.11   7.28    -28      12      19      23     29  
WB_mean   3,801,750 20.69   6.26   -18.79  16.67   22.38   25.54   31.08
WB_max    3,801,750 24.64   5.47    -13      22      26      29     37  
------------------------------------------------------------------------


# Categorize T mean/max in the days preceding measurement into bins

In [5]:
thresholds_WB_min <- c(-28,10,12,14,16,18,20,22,24,37)
thresholds_WB_mean <- c(-19,14,16,18,20,22,24,26,28,37)
thresholds_WB_max <- c(-19,18,20,22,24,26,28,30,32,37)

In [6]:
df_deg_bins_day_bins <- df_deg_bins_day %>%
                            mutate(WB_min_bin = cut(WB_min, breaks = thresholds_WB_min, 
                                                 labels = c("WB_MinBin_inf10","WB_MinBin_10_12",
                                                            "WB_MinBin_12_14","WB_MinBin_14_16",
                                                            "WB_MinBin_16_18","WB_MinBin_18_20",
                                                            "WB_MinBin_20_22","WB_MinBin_22_24",
                                                            "WB_MinBin_sup24"),
                                                 include.lowest = TRUE),
                                   WB_mean_bin = cut(WB_mean, breaks = thresholds_WB_mean, 
                                                 labels = c("WB_MeanBin_inf14","WB_MeanBin_14_16",
                                                            "WB_MeanBin_16_18","WB_MeanBin_18_20",
                                                            "WB_MeanBin_20_22","WB_MeanBin_22_24",
                                                            "WB_MeanBin_24_26","WB_MeanBin_26_28","WB_MeanBin_sup28"),
                                                 include.lowest = TRUE),
                                   WB_max_bin = cut(WB_max, breaks = thresholds_WB_max, 
                                                 labels = c("WB_MaxBin_inf18","WB_MaxBin_18_20","WB_MaxBin_20_22",
                                                            "WB_MaxBin_22_24","WB_MaxBin_24_26",
                                                            "WB_MaxBin_26_28","WB_MaxBin_28_30",
                                                            "WB_MaxBin_30_32","WB_MaxBin_sup32"),
                                                 include.lowest = TRUE))
sprintf("%i x %i dataframe", nrow(df_deg_bins_day_bins), ncol(df_deg_bins_day_bins))
head(df_deg_bins_day_bins,2)

[1] "3801750 x 10 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,WB_min,WB_mean,WB_max,WB_min_bin,WB_mean_bin,WB_max_bin
<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<fct>,<fct>,<fct>
42022,74,15.25,1,19,22.00000,26,WB_MinBin_18_20,WB_MeanBin_20_22,WB_MaxBin_24_26
42022,74,15.25,2,18,21.45833,25,WB_MinBin_16_18,WB_MeanBin_20_22,WB_MaxBin_24_26


In [7]:
rm(df_deg_bins,df_deg_bins_day)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1556339,83.2,27634961,1475.9,34543701,1844.9
Vcells,29699893,226.6,388987119,2967.8,485689482,3705.6


# Pivot wider

## Min temperature

In [8]:
df_deg_min_wide <- df_deg_bins_day_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,WB_min_bin)%>%
                        mutate(Value=1)%>%
                        pivot_wider(names_from = WB_min_bin, values_from = Value)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_min_wide), ncol(df_deg_min_wide))
head(df_deg_min_wide,2)

[1] "3801750 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,WB_MinBin_18_20,WB_MinBin_16_18,WB_MinBin_14_16,WB_MinBin_12_14,WB_MinBin_10_12,WB_MinBin_inf10,WB_MinBin_20_22,WB_MinBin_22_24,WB_MinBin_sup24
<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,1,1,0,0,0,0,0,0,0,0
42022,74,15.25,2,0,1,0,0,0,0,0,0,0


In [9]:
df_deg_min_Ndays_30 <- df_deg_min_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_min_Ndays_30), ncol(df_deg_min_Ndays_30))
head(df_deg_min_Ndays_30,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MinBin_18_20_Ndays_on30,WB_MinBin_16_18_Ndays_on30,WB_MinBin_14_16_Ndays_on30,WB_MinBin_12_14_Ndays_on30,WB_MinBin_10_12_Ndays_on30,WB_MinBin_inf10_Ndays_on30,WB_MinBin_20_22_Ndays_on30,WB_MinBin_22_24_Ndays_on30,WB_MinBin_sup24_Ndays_on30
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,13,11,4,2,0,0,0,0,0
42022,74,15.50,1,8,11,5,5,0,0,0,0


In [10]:
df_deg_min_Ndays_20 <- df_deg_min_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_min_Ndays_20), ncol(df_deg_min_Ndays_20))
head(df_deg_min_Ndays_20,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MinBin_18_20_Ndays_on20,WB_MinBin_16_18_Ndays_on20,WB_MinBin_14_16_Ndays_on20,WB_MinBin_12_14_Ndays_on20,WB_MinBin_10_12_Ndays_on20,WB_MinBin_inf10_Ndays_on20,WB_MinBin_20_22_Ndays_on20,WB_MinBin_22_24_Ndays_on20,WB_MinBin_sup24_Ndays_on20
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,8,6,4,2,0,0,0,0,0
42022,74,15.50,1,7,4,3,5,0,0,0,0


In [11]:
df_deg_min_Ndays_10 <- df_deg_min_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_min_Ndays_10), ncol(df_deg_min_Ndays_10))
head(df_deg_min_Ndays_10,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MinBin_18_20_Ndays_on10,WB_MinBin_16_18_Ndays_on10,WB_MinBin_14_16_Ndays_on10,WB_MinBin_12_14_Ndays_on10,WB_MinBin_10_12_Ndays_on10,WB_MinBin_inf10_Ndays_on10,WB_MinBin_20_22_Ndays_on10,WB_MinBin_22_24_Ndays_on10,WB_MinBin_sup24_Ndays_on10
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,1,3,4,2,0,0,0,0,0
42022,74,15.50,0,1,1,3,5,0,0,0,0


In [12]:
rm(df_deg_min_wide)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1571985,84.0,22107969,1180.7,34543701,1844.9
Vcells,34102487,260.2,311189696,2374.2,485689482,3705.6


## Mean temperature

In [13]:
df_deg_mean_wide <- df_deg_bins_day_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,WB_mean_bin)%>%
                        mutate(Value=1)%>%
                        pivot_wider(names_from = WB_mean_bin, values_from = Value)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_mean_wide), ncol(df_deg_mean_wide))
head(df_deg_mean_wide,2)

[1] "3801750 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,WB_MeanBin_20_22,WB_MeanBin_18_20,WB_MeanBin_22_24,WB_MeanBin_16_18,WB_MeanBin_inf14,WB_MeanBin_14_16,WB_MeanBin_24_26,WB_MeanBin_26_28,WB_MeanBin_sup28
<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,1,1,0,0,0,0,0,0,0,0
42022,74,15.25,2,1,0,0,0,0,0,0,0,0


In [14]:
df_deg_mean_Ndays_30 <- df_deg_mean_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_mean_Ndays_30), ncol(df_deg_mean_Ndays_30))
head(df_deg_mean_Ndays_30,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MeanBin_20_22_Ndays_on30,WB_MeanBin_18_20_Ndays_on30,WB_MeanBin_22_24_Ndays_on30,WB_MeanBin_16_18_Ndays_on30,WB_MeanBin_inf14_Ndays_on30,WB_MeanBin_14_16_Ndays_on30,WB_MeanBin_24_26_Ndays_on30,WB_MeanBin_26_28_Ndays_on30,WB_MeanBin_sup28_Ndays_on30
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,18,4,8,0,0,0,0,0,0
42022,74,15.50,15,6,5,4,0,0,0,0,0


In [15]:
df_deg_mean_Ndays_20 <- df_deg_mean_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_mean_Ndays_20), ncol(df_deg_mean_Ndays_20))
head(df_deg_mean_Ndays_20,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MeanBin_20_22_Ndays_on20,WB_MeanBin_18_20_Ndays_on20,WB_MeanBin_22_24_Ndays_on20,WB_MeanBin_16_18_Ndays_on20,WB_MeanBin_inf14_Ndays_on20,WB_MeanBin_14_16_Ndays_on20,WB_MeanBin_24_26_Ndays_on20,WB_MeanBin_26_28_Ndays_on20,WB_MeanBin_sup28_Ndays_on20
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,11,4,5,0,0,0,0,0,0
42022,74,15.50,9,3,4,4,0,0,0,0,0


In [16]:
df_deg_mean_Ndays_10 <- df_deg_mean_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_mean_Ndays_10), ncol(df_deg_mean_Ndays_10))
head(df_deg_mean_Ndays_10,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MeanBin_20_22_Ndays_on10,WB_MeanBin_18_20_Ndays_on10,WB_MeanBin_22_24_Ndays_on10,WB_MeanBin_16_18_Ndays_on10,WB_MeanBin_inf14_Ndays_on10,WB_MeanBin_14_16_Ndays_on10,WB_MeanBin_24_26_Ndays_on10,WB_MeanBin_26_28_Ndays_on10,WB_MeanBin_sup28_Ndays_on10
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,6,4,0,0,0,0,0,0,0
42022,74,15.50,3,3,0,4,0,0,0,0,0


In [17]:
rm(df_deg_mean_wide)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1572157,84.0,14149101,755.7,34543701,1844.9
Vcells,38474731,293.6,248951757,1899.4,485689482,3705.6


## Max temperature

In [18]:
df_deg_max_wide <- df_deg_bins_day_bins %>%
                        select(Measured_date_CDC,cell_x,cell_y,Day_Min,WB_max_bin)%>%
                        mutate(Value=1)%>%
                        pivot_wider(names_from = WB_max_bin, values_from = Value)%>%
                        ungroup()%>%
                        mutate_all(~replace_na(., 0))
sprintf("%i x %i dataframe", nrow(df_deg_max_wide), ncol(df_deg_max_wide))
head(df_deg_max_wide,2)

[1] "3801750 x 13 dataframe"

Measured_date_CDC,cell_x,cell_y,Day_Min,WB_MaxBin_24_26,WB_MaxBin_22_24,WB_MaxBin_26_28,WB_MaxBin_inf18,WB_MaxBin_18_20,WB_MaxBin_20_22,WB_MaxBin_28_30,WB_MaxBin_30_32,WB_MaxBin_sup32
<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,1,1,0,0,0,0,0,0,0,0
42022,74,15.25,2,1,0,0,0,0,0,0,0,0


In [19]:
df_deg_max_Ndays_30 <- df_deg_max_wide %>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on30 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_max_Ndays_30), ncol(df_deg_max_Ndays_30))
head(df_deg_max_Ndays_30,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MaxBin_24_26_Ndays_on30,WB_MaxBin_22_24_Ndays_on30,WB_MaxBin_26_28_Ndays_on30,WB_MaxBin_inf18_Ndays_on30,WB_MaxBin_18_20_Ndays_on30,WB_MaxBin_20_22_Ndays_on30,WB_MaxBin_28_30_Ndays_on30,WB_MaxBin_30_32_Ndays_on30,WB_MaxBin_sup32_Ndays_on30
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,16,12,2,0,0,0,0,0,0
42022,74,15.50,18,7,5,0,0,0,0,0,0


In [20]:
df_deg_max_Ndays_20 <- df_deg_max_wide %>%
                        filter(Day_Min <= 20)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on20 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_max_Ndays_20), ncol(df_deg_max_Ndays_20))
head(df_deg_max_Ndays_20,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MaxBin_24_26_Ndays_on20,WB_MaxBin_22_24_Ndays_on20,WB_MaxBin_26_28_Ndays_on20,WB_MaxBin_inf18_Ndays_on20,WB_MaxBin_18_20_Ndays_on20,WB_MaxBin_20_22_Ndays_on20,WB_MaxBin_28_30_Ndays_on20,WB_MaxBin_30_32_Ndays_on20,WB_MaxBin_sup32_Ndays_on20
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,9,10,1,0,0,0,0,0,0
42022,74,15.50,12,7,1,0,0,0,0,0,0


In [21]:
df_deg_max_Ndays_10 <- df_deg_max_wide %>%
                        filter(Day_Min <= 10)%>%
                        select(-Day_Min)%>%
                        group_by(Measured_date_CDC,cell_x,cell_y)%>%
                        summarise_all(list(Ndays_on10 =~ sum(.)))%>%
                        ungroup()
sprintf("%i x %i dataframe", nrow(df_deg_max_Ndays_10), ncol(df_deg_max_Ndays_10))
head(df_deg_max_Ndays_10,2)

[1] "126725 x 12 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_MaxBin_24_26_Ndays_on10,WB_MaxBin_22_24_Ndays_on10,WB_MaxBin_26_28_Ndays_on10,WB_MaxBin_inf18_Ndays_on10,WB_MaxBin_18_20_Ndays_on10,WB_MaxBin_20_22_Ndays_on10,WB_MaxBin_28_30_Ndays_on10,WB_MaxBin_30_32_Ndays_on10,WB_MaxBin_sup32_Ndays_on10
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
42022,74,15.25,4,6,0,0,0,0,0,0,0
42022,74,15.50,5,5,0,0,0,0,0,0,0


In [22]:
rm(df_deg_max_wide)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1572329,84.0,9055425,483.7,34543701,1844.9
Vcells,42846973,326.9,199161406,1519.5,485689482,3705.6


# Merge

In [23]:
result <- data.frame(df_deg_mean_Ndays_30) %>%
                    left_join(df_deg_mean_Ndays_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_mean_Ndays_10, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_min_Ndays_30, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_min_Ndays_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_min_Ndays_10, by = c("Measured_date_CDC","cell_x","cell_y"))%>%                    left_join(df_deg_max_Ndays_30, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_max_Ndays_20, by = c("Measured_date_CDC","cell_x","cell_y"))%>%
                    left_join(df_deg_max_Ndays_10, by = c("Measured_date_CDC","cell_x","cell_y"))
sprintf("%i x %i dataframe", nrow(result), ncol(result))
head(result,2)

[1] "126725 x 84 dataframe"

,Measured_date_CDC,cell_x,cell_y,WB_MeanBin_20_22_Ndays_on30,WB_MeanBin_18_20_Ndays_on30,WB_MeanBin_22_24_Ndays_on30,WB_MeanBin_16_18_Ndays_on30,WB_MeanBin_inf14_Ndays_on30,WB_MeanBin_14_16_Ndays_on30,WB_MeanBin_24_26_Ndays_on30,WB_MeanBin_26_28_Ndays_on30,WB_MeanBin_sup28_Ndays_on30,WB_MeanBin_20_22_Ndays_on20,WB_MeanBin_18_20_Ndays_on20,WB_MeanBin_22_24_Ndays_on20,WB_MeanBin_16_18_Ndays_on20,WB_MeanBin_inf14_Ndays_on20,WB_MeanBin_14_16_Ndays_on20,WB_MeanBin_24_26_Ndays_on20,WB_MeanBin_26_28_Ndays_on20,WB_MeanBin_sup28_Ndays_on20,WB_MeanBin_20_22_Ndays_on10,WB_MeanBin_18_20_Ndays_on10,WB_MeanBin_22_24_Ndays_on10,WB_MeanBin_16_18_Ndays_on10,WB_MeanBin_inf14_Ndays_on10,WB_MeanBin_14_16_Ndays_on10,WB_MeanBin_24_26_Ndays_on10,WB_MeanBin_26_28_Ndays_on10,WB_MeanBin_sup28_Ndays_on10,WB_MinBin_18_20_Ndays_on30,WB_MinBin_16_18_Ndays_on30,WB_MinBin_14_16_Ndays_on30,WB_MinBin_12_14_Ndays_on30,WB_MinBin_10_12_Ndays_on30,WB_MinBin_inf10_Ndays_on30,WB_MinBin_20_22_Ndays_on30,WB_MinBin_22_24_Ndays_on30,WB_MinBin_sup24_Ndays_on30,WB_MinBin_18_20_Ndays_on20,WB_MinBin_16_18_Ndays_on20,WB_MinBin_14_16_Ndays_on20,WB_MinBin_12_14_Ndays_on20,WB_MinBin_10_12_Ndays_on20,WB_MinBin_inf10_Ndays_on20,WB_MinBin_20_22_Ndays_on20,WB_MinBin_22_24_Ndays_on20,WB_MinBin_sup24_Ndays_on20,WB_MinBin_18_20_Ndays_on10,WB_MinBin_16_18_Ndays_on10,WB_MinBin_14_16_Ndays_on10,WB_MinBin_12_14_Ndays_on10,WB_MinBin_10_12_Ndays_on10,WB_MinBin_inf10_Ndays_on10,WB_MinBin_20_22_Ndays_on10,WB_MinBin_22_24_Ndays_on10,WB_MinBin_sup24_Ndays_on10,WB_MaxBin_24_26_Ndays_on30,WB_MaxBin_22_24_Ndays_on30,WB_MaxBin_26_28_Ndays_on30,WB_MaxBin_inf18_Ndays_on30,WB_MaxBin_18_20_Ndays_on30,WB_MaxBin_20_22_Ndays_on30,WB_MaxBin_28_30_Ndays_on30,WB_MaxBin_30_32_Ndays_on30,WB_MaxBin_sup32_Ndays_on30,WB_MaxBin_24_26_Ndays_on20,WB_MaxBin_22_24_Ndays_on20,WB_MaxBin_26_28_Ndays_on20,WB_MaxBin_inf18_Ndays_on20,WB_MaxBin_18_20_Ndays_on20,WB_MaxBin_20_22_Ndays_on20,WB_MaxBin_28_30_Ndays_on20,WB_MaxBin_30_32_Ndays_on20,WB_MaxBin_sup32_Ndays_on20,WB_MaxBin_24_26_Ndays_on10,WB_MaxBin_22_24_Ndays_on10,WB_MaxBin_26_28_Ndays_on10,WB_MaxBin_inf18_Ndays_on10,WB_MaxBin_18_20_Ndays_on10,WB_MaxBin_20_22_Ndays_on10,WB_MaxBin_28_30_Ndays_on10,WB_MaxBin_30_32_Ndays_on10,WB_MaxBin_sup32_Ndays_on10
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,42022,74,15.25,18,4,8,0,0,0,0,0,0,11,4,5,0,0,0,0,0,0,6,4,0,0,0,0,0,0,0,13,11,4,2,0,0,0,0,0,8,6,4,2,0,0,0,0,0,1,3,4,2,0,0,0,0,0,16,12,2,0,0,0,0,0,0,9,10,1,0,0,0,0,0,0,4,6,0,0,0,0,0,0,0
2,42022,74,15.50,15,6,5,4,0,0,0,0,0,9,3,4,4,0,0,0,0,0,3,3,0,4,0,0,0,0,0,1,8,11,5,5,0,0,0,0,1,7,4,3,5,0,0,0,0,0,1,1,3,5,0,0,0,0,18,7,5,0,0,0,0,0,0,12,7,1,0,0,0,0,0,0,5,5,0,0,0,0,0,0,0


# Save

In [24]:
fwrite(result,"./output/df_WB_cells_CDC_Ndays.csv")